# 🎵 Music Genre Classification using CNN

## Objective
Predict the genre of a music audio file using **MFCC (Mel-Frequency Cepstral Coefficients)** features and a **Convolutional Neural Network (CNN)**.

### Tools & Technologies
- Python
- Librosa
- NumPy
- Scikit-learn
- TensorFlow / Keras
- Matplotlib

### Genres
Blues, Classical, Country, Disco, Hip-Hop, Jazz, Metal, Pop, Reggae, Rock


## 1. Import Required Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical


## 2. Define Dataset and Feature Parameters

In [ ]:
DATASET_PATH = "../dataset"

GENRES = [
    "blues",
    "classical",
    "country",
    "disco",
    "hiphop",
    "jazz",
    "metal",
    "pop",
    "reggae",
    "rock"
]

SAMPLE_RATE = 22050
DURATION = 30
N_MFCC = 40
MAX_PAD_LEN = 130

print("Genres:", GENRES)


## 3. Extract MFCC Features

MFCCs represent important characteristics of an audio signal in a compact form that can be used by the CNN for genre classification.


In [ ]:
def extract_mfcc(file_path):
    try:
        audio, sr = librosa.load(
            file_path,
            sr=SAMPLE_RATE,
            duration=DURATION
        )

        mfcc = librosa.feature.mfcc(
            y=audio,
            sr=sr,
            n_mfcc=N_MFCC
        )

        if mfcc.shape[1] < MAX_PAD_LEN:
            pad_width = MAX_PAD_LEN - mfcc.shape[1]
            mfcc = np.pad(
                mfcc,
                pad_width=((0, 0), (0, pad_width)),
                mode="constant"
            )
        else:
            mfcc = mfcc[:, :MAX_PAD_LEN]

        return mfcc

    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None


## 4. Load the Audio Dataset

In [ ]:
X = []
y = []

print("Loading audio files...")

for genre in GENRES:
    genre_path = os.path.join(DATASET_PATH, genre)

    if not os.path.exists(genre_path):
        print(f"Warning: {genre_path} not found.")
        continue

    for filename in os.listdir(genre_path):
        if filename.lower().endswith(".wav"):
            file_path = os.path.join(genre_path, filename)
            mfcc = extract_mfcc(file_path)

            if mfcc is not None:
                X.append(mfcc)
                y.append(genre)

print("Total audio files loaded:", len(X))


In [ ]:
X = np.array(X)
y = np.array(y)

X = X[..., np.newaxis]

print("Feature shape:", X.shape)
print("Label shape:", y.shape)


## 5. Encode Genre Labels and Split the Dataset

The dataset is divided into:
- **80% training data**
- **20% validation data**


In [ ]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)
y_encoded = to_categorical(
    y_encoded,
    num_classes=len(GENRES)
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_test))


## 6. Build the CNN Model

In [ ]:
model = Sequential([
    Conv2D(
        32,
        (3, 3),
        activation="relu",
        input_shape=(X.shape[1], X.shape[2], 1)
    ),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),

    Conv2D(128, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),

    Flatten(),

    Dense(128, activation="relu"),
    Dropout(0.3),

    Dense(len(GENRES), activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


## 7. Train the CNN Model

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=32
)


## 8. Evaluate the Model

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)

print(f"Validation Loss: {loss:.4f}")
print(f"Validation Accuracy: {accuracy * 100:.2f}%")


## 9. Plot Training and Validation Accuracy

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.show()


## 10. Save the Trained Model

In [ ]:
MODEL_PATH = "../models/music_genre_cnn_notebook.keras"

os.makedirs("../models", exist_ok=True)

model.save(MODEL_PATH)

print("Model saved to:", MODEL_PATH)


## 11. Predict the Genre of an Audio File

In [ ]:
def predict_genre(audio_file):
    mfcc = extract_mfcc(audio_file)

    if mfcc is None:
        print("Could not process audio file.")
        return

    mfcc = mfcc[np.newaxis, ..., np.newaxis]

    probabilities = model.predict(
        mfcc,
        verbose=0
    )[0]

    predicted_index = np.argmax(probabilities)
    predicted_genre = GENRES[predicted_index]
    confidence = probabilities[predicted_index] * 100

    print("Predicted Genre:", predicted_genre.upper())
    print(f"Confidence: {confidence:.2f}%")

    return predicted_genre, confidence


### Example Prediction

In [ ]:
audio_file = "../dataset/blues/blues_000.wav"

predict_genre(audio_file)


## 12. Display MFCC Visualization

This visualization shows the MFCC representation extracted from an audio file.


In [ ]:
audio_file = "../dataset/blues/blues_000.wav"

audio, sr = librosa.load(
    audio_file,
    sr=SAMPLE_RATE
)

mfcc = librosa.feature.mfcc(
    y=audio,
    sr=sr,
    n_mfcc=N_MFCC
)

plt.figure(figsize=(10, 5))

librosa.display.specshow(
    mfcc,
    x_axis="time",
    sr=sr
)

plt.colorbar()
plt.title("MFCC Representation")
plt.tight_layout()
plt.show()


## 13. Project Result

The CNN model successfully classifies uploaded music audio into one of the ten GTZAN genres.

For the current demonstration dataset of **100 audio clips**, the trained model achieved approximately **70% validation accuracy**.

The project also includes a Streamlit web application where users can upload an audio file and receive:
- Predicted music genre
- Confidence score
- Genre probability chart

> Note: The current dataset is a small demonstration subset, so the validation accuracy should not be treated as a measure of performance on the full GTZAN dataset.
